## Step 1 — Load and pivot the juror votes

**Input:** `jury_votes.xlsx` in long format with columns `Voting_country`, `Juror`, `Participating_country`, `Rank`.

**Output (`ranks_df`):** a wide DataFrame where:
- rows are **participating countries** (in source order),
- columns are a 2-level `MultiIndex` `(Voting_country, Juror)`, grouped by voting country.

Each cell holds the rank that juror gave to that participating country. A `0` means the juror is from that country (self-vote).

In [53]:
import pandas as pd

In [54]:
INPUT_FILE = r'C:\Users\PS383WL\EY\Engagement - Starlight - ESC 2026 - Vienna\Technical Assessment\1 Scripts\2 Armand Test Folder\01. SF1 - Jury\jury_vote_sf1.xlsx'

long_df = pd.read_excel(INPUT_FILE)

DICT_ISO = {
    "Albania": "AL", "Andorra": "AD", "Armenia": "AM", "Australia": "AU",
    "Austria": "AT", "Azerbaijan": "AZ", "Belarus": "BY", "Belgium": "BE",
    "Bosnia & Herzegovina": "BA", "Bulgaria": "BG", "Croatia": "HR",
    "Cyprus": "CY", "Czechia": "CZ", "Denmark": "DK", "Estonia": "EE",
    "Finland": "FI", "France": "FR", "Georgia": "GE", "Germany": "DE",
    "Greece": "GR", "Hungary": "HU", "Iceland": "IS", "Ireland": "IE",
    "Israel": "IL", "Italy": "IT", "Latvia": "LV", "Lithuania": "LT",
    "Luxembourg": "LU", "Malta": "MT", "Moldova": "MD", "Monaco": "MC",
    "Montenegro": "ME", "Morocco": "MA", "Netherlands": "NL",
    "North Macedonia": "MK", "Norway": "NO", "Poland": "PL",
    "Portugal": "PT", "Romania": "RO", "Russia": "RU", "San Marino": "SM",
    "Serbia": "RS", "Slovakia": "SK", "Slovenia": "SI", "Spain": "ES",
    "Sweden": "SE", "Switzerland": "CH", "Turkey": "TR", "Ukraine": "UA",
    "United Kingdom": "GB", "Rest Of World": "RoW",
}

# Invert the dict for ISO -> full name lookup
ISO_TO_COUNTRY = {iso: name for name, iso in DICT_ISO.items()}

# Validate every ISO in the data has a known mapping (fail fast if not)
for col in ['Voting_country', 'Participating_country']:
    unknown = set(long_df[col]) - set(ISO_TO_COUNTRY)
    if unknown:
        raise ValueError(f'Unknown ISO codes in {col}: {sorted(unknown)}')
    long_df[col] = long_df[col].map(ISO_TO_COUNTRY)

long_df.head()

,Voting_country,Juror,Participating_country,Rank
0,Croatia,Juror 1,Croatia,0
1,Croatia,Juror 2,Croatia,0
2,Croatia,Juror 3,Croatia,0
3,Croatia,Juror 4,Croatia,0
4,Croatia,Juror 5,Croatia,0


In [55]:
# Capture source order so the pivot keeps it (pandas would otherwise sort alphabetically)
participating_order = long_df['Participating_country'].drop_duplicates().tolist()
voting_order        = long_df['Voting_country'].drop_duplicates().tolist()
juror_order         = long_df['Juror'].drop_duplicates().tolist()

ranks_df = (
    long_df
    .pivot(index='Participating_country',
           columns=['Voting_country', 'Juror'],
           values='Rank')
    .reindex(index=participating_order,
             columns=pd.MultiIndex.from_product(
                 [voting_order, juror_order],
                 names=['Voting_country', 'Juror']))
)

ranks_df

Voting_country        Croatia                                                  \
Juror                 Juror 1 Juror 2 Juror 3 Juror 4 Juror 5 Juror 6 Juror 7   
Participating_country                                                           
Croatia                     0       0       0       0       0       0       0   
Finland                     9       3       8       7       5      11       1   
Montenegro                 13       6       9      14       9       1       7   
Serbia                      8       8       4       3       7       4       6   
Sweden                      7      10      12      12       3       9       2   
Belgium                    14       7      11       4       8      13       5   
Georgia                    12      12      14       9      13      12      13   
Israel                      6       5       2      11       1      10      14   
Moldova                     3      13      13       8       2       3       8   
Poland                     10       9       6       1       4       6      12   
Estonia                     4      11       7       5      12       8       4   
Greece                      5       4       3      10       6       5       9   
Lithuania                   1       2       1      13      10       7       3   
Portugal                    2      14       5       6      11       2      11   
San Marino                 11       1      10       2      14      14      10   

Voting_country        Finland                  ... Germany                  \
Juror                 Juror 1 Juror 2 Juror 3  ... Juror 5 Juror 6 Juror 7   
Participating_country                          ...                           
Croatia                    11       1      11  ...       5      12      12   
Finland                     0       0       0  ...       3      14       2   
Montenegro                  5       2       5  ...      15      15      14   
Serbia                      9      12      14  ...       1       5       8   
Sweden                      8       9       2  ...       4      13       3   
Belgium                     2      14      12  ...       8       1       6   
Georgia                    10      10      13  ...      13       7       5   
Israel                      3       6       1  ...      12      11       1   
Moldova                     7       4       8  ...       7       8      11   
Poland                      6      13       9  ...       2       3      15   
Estonia                    12       5       3  ...      11       4       9   
Greece                      4       3       4  ...       6       9       7   
Lithuania                   1       8       6  ...      14      10      13   
Portugal                   13      11       7  ...       9       2       4   
San Marino                 14       7      10  ...      10       6      10   

Voting_country          Italy                                                  
Juror                 Juror 1 Juror 2 Juror 3 Juror 4 Juror 5 Juror 6 Juror 7  
Participating_country                                                          
Croatia                     2      10      11       1       9       6       5  
Finland                     8       1       9       8      12      14      12  
Montenegro                  7      14      12      12       6       2      14  
Serbia                     10       5       8      13       8      15       3  
Sweden                      4      13       7      14       3      10       4  
Belgium                    11       3       6      10      13       9      13  
Georgia                    14      15       4       3       2       5      11  
Israel                      9       6       5      11       5       4       9  
Moldova                     3       8       2      15      14      13      15  
Poland                      1      11      14       6      11      11      10  
Estonia                    15       7      10       2      15       3       6  
Greece          

In [56]:
# Quick sanity checks
print('Shape :', ranks_df.shape)
print('NaNs  :', ranks_df.isna().sum().sum(), '(should be 0)')
print('Self-votes (rank 0) per voting country:')
(ranks_df == 0).sum().groupby(level='Voting_country').sum()

Shape : (15, 119)
NaNs  : 0 (should be 0)
Self-votes (rank 0) per voting country:


Voting_country
Belgium       7
Croatia       7
Estonia       7
Finland       7
Georgia       7
Germany       0
Greece        7
Israel        7
Italy         0
Lithuania     7
Moldova       7
Montenegro    7
Poland        7
Portugal      7
San Marino    7
Serbia        7
Sweden        7
dtype: int64

## Step 2 — Convert ranks to exponential scores

Each juror's rank is mapped to an exponential point value via a fixed lookup table
(rank 1 → 12, decreasing to rank 26 → 0.10382). A self-vote (rank 0) contributes 0.

In [57]:
class RankError(Exception):
    """Raised when a rank value isn't in the supported mapping range (0–26)."""
    pass


EXP_SCORE_TABLE = {
    0:  0,        1:  12,       2:  9.92351,  3:  8.20634,  4:  6.78631,
    5:  5.612,    6:  4.64089,  7:  3.83783,  8:  3.17373,  9:  2.62454,
    10: 2.17039, 11: 1.79482, 12: 1.48425, 13: 1.22741, 14: 1.01502,
    15: 0.83938, 16: 0.69413, 17: 0.57402, 18: 0.47469, 19: 0.39255,
    20: 0.32462, 21: 0.26845, 22: 0.222,   23: 0.18358, 24: 0.15181,
    25: 0.12554, 26: 0.10382,
}

def rank_to_exp_score(rank):
    """Map a juror rank to its exponential point value."""
    if rank not in EXP_SCORE_TABLE:
        raise RankError(
            f'Invalid Rank Value Provided for Exp. Scoring Mapping: {rank}'
        )
    return EXP_SCORE_TABLE[rank]

In [58]:
# Element-wise map preserves the (Voting_country, Juror) MultiIndex columns
# and the Participating_country index.
exp_scores_df = ranks_df.map(rank_to_exp_score)
exp_scores_df

Voting_country          Croatia                                          \
Juror                   Juror 1   Juror 2   Juror 3   Juror 4   Juror 5   
Participating_country                                                     
Croatia                 0.00000   0.00000   0.00000   0.00000   0.00000   
Finland                 2.62454   8.20634   3.17373   3.83783   5.61200   
Montenegro              1.22741   4.64089   2.62454   1.01502   2.62454   
Serbia                  3.17373   3.17373   6.78631   8.20634   3.83783   
Sweden                  3.83783   2.17039   1.48425   1.48425   8.20634   
Belgium                 1.01502   3.83783   1.79482   6.78631   3.17373   
Georgia                 1.48425   1.48425   1.01502   2.62454   1.22741   
Israel                  4.64089   5.61200   9.92351   1.79482  12.00000   
Moldova                 8.20634   1.22741   1.22741   3.17373   9.92351   
Poland                  2.17039   2.62454   4.64089  12.00000   6.78631   
Estonia                 6.78631   1.79482   3.83783   5.61200   1.48425   
Greece                  5.61200   6.78631   8.20634   2.17039   4.64089   
Lithuania              12.00000   9.92351  12.00000   1.22741   2.17039   
Portugal                9.92351   1.01502   5.61200   4.64089   1.79482   
San Marino              1.79482  12.00000   2.17039   9.92351   1.01502   

Voting_country                              Finland                      ...  \
Juror                   Juror 6   Juror 7   Juror 1   Juror 2   Juror 3  ...   
Participating_country                                                    ...   
Croatia                 0.00000   0.00000   1.79482  12.00000   1.79482  ...   
Finland                 1.79482  12.00000   0.00000   0.00000   0.00000  ...   
Montenegro             12.00000   3.83783   5.61200   9.92351   5.61200  ...   
Serbia                  6.78631   4.64089   2.62454   1.48425   1.01502  ...   
Sweden                  2.62454   9.92351   3.17373   2.62454   9.92351  ...   
Belgium                 1.22741   5.61200   9.92351   1.01502   1.48425  ...   
Georgia                 1.48425   1.22741   2.17039   2.17039   1.22741  ...   
Israel                  2.17039   1.01502   8.20634   4.64089  12.00000  ...   
Moldova                 8.20634   3.17373   3.83783   6.78631   3.17373  ...   
Poland                  4.64089   1.48425   4.64089   1.22741   2.62454  ...   
Estonia                 3.17373   6.78631   1.48425   5.61200   8.20634  ...   
Greece                  5.61200   2.62454   6.78631   8.20634   6.78631  ...   
Lithuania               3.83783   8.20634  12.00000   3.17373   4.64089  ...   
Portugal                9.92351   1.79482   1.22741   1.79482   3.83783  ...   
San Marino              1.01502   2.17039   1.01502   3.83783   2.17039  ...   

Voting_country          Germany                         Italy            \
Juror                   Juror 5   Juror 6   Juror 7   Juror 1   Juror 2   
Participating_country                                                     
Croatia                 5.61200   1.48425   1.48425   9.92351   2.17039   
Finland                 8.20634   1.01502   9.92351   3.17373  12.00000   
Montenegro              0.83938   0.83938   1.01502   3.83783   1.01502   
Serbia                 12.00000   5.61200   3.17373   2.17039   5.61200   
Sweden                  6.78631   1.22741   8.20634   6.78631   1.22741   
Belgium                 3.17373  12.00000   4.64089   1.79482   8.20634   
Georgia                 1.22741   3.83783   5.61200   1.01502   0.83938   
Israel                  1.48425   1.79482  12.00000   2.62454   4.64089   
Moldova                 3.83783   3.17373   1.79482   8.20634   3.17373   
Poland                  9.92351   8.20634   0.83938  12.00000   1.79482   
Estonia                 1.79482   6.78631   2.62454   0.83938   3.83783   
Greece                  4.64089   2.62454   3.83783   4.64089   6.78631   
Lithuania               1.01502   2.17039   1.22741   5.61200   2.62454   
Portugal

## Step 3 — Sum per jury, rank, award points

For each national jury:
1. Sum the exponential scores across its jurors → one value per participating country.
2. Mask the home country (a jury never awards points to itself).
3. Rank the remaining countries by sum (highest = rank 1).
4. Convert rank → points using the 12-10-8-7-6-5-4-3-2-1 scale.

Ties (where two countries share a rank) are detected and stored in `ties_found` for the
tie-breaking step that comes later.

In [59]:
# 3a — Sum exponential scores per (Voting_country, Participating_country)
# Collapses the 'Juror' level by summing across jurors of each voting country.
jury_sums = (
    exp_scores_df
    .T.groupby(level='Voting_country', sort=False).sum().T
)

jury_sums

Voting_country,Croatia,Finland,Montenegro,Serbia,Sweden,Belgium,Georgia,Israel,Moldova,Poland,Estonia,Greece,Lithuania,Portugal,San Marino,Germany,Italy
Participating_country,,,,,,,,,,,,,,,,,
Croatia,0.00000,30.92931,45.85070,25.89900,32.72267,40.86299,45.85703,32.51879,44.65932,23.15477,36.42352,40.79219,33.89326,25.53099,15.95822,22.17314,38.76615
Finland,37.24926,0.00000,33.70086,35.24254,37.30812,36.76560,15.46817,40.12413,43.67239,39.89073,26.66754,27.60791,39.22663,39.05745,30.75389,59.55755,24.95552
Montenegro,27.97023,50.55796,0.00000,27.23270,24.20920,52.50187,41.90968,18.77400,43.78368,30.02885,40.15467,39.36802,29.87671,28.94176,29.85641,23.67322,23.40077
Serbia,36.60514,25.72991,37.88314,0.00000,20.27267,35.60626,31.46772,42.43925,32.56918,44.45397,31.69390,38.49422,23.66165,36.25428,54.26661,30.04695,24.40298
Sweden,29.73111,33.94126,36.08848,31.23649,0.00000,33.80401,26.91127,33.28131,34.67964,30.88708,18.38850,30.76813,23.83795,37.61951,24.90045,51.05094,30.02961
Belgium,23.44712,25.56823,14.67015,27.53971,38.93245,0.00000,40.05538,36.82616,20.14665,33.56325,20.69828,26.07858,42.11382,28.81896,23.68788,30.71961,21.89180
Georgia,10.54713,29.03956,25.32136,33.88090,20.90786,35.13430,0.00000,28.32497,35.31776,39.66272,34.35302,39.08427,38.45126,18.17036,35.92278,30.15149,34.17738
Israel,37.15663,48.46745,23.04250,26.41217,32.08468,23.29244,16.97707,0.00000,27.31964,28.94034,35.39784,40.92622,18.39893,25.31139,16.59185,41.93737,29.69510
Moldova,35.13847,34.16054,26.87603,65.98530,30.07302,29.21743,18.25632,27.58198,0.00000,35.13412,36.67395,28.47272,28.87138,28.81903,61.17735,17.82437,25.22477


In [60]:
# 3b — Mask the home country before ranking (a jury doesn't rank itself)
jury_sums_for_ranking = jury_sums.copy()
for vc in jury_sums_for_ranking.columns:
    if vc in jury_sums_for_ranking.index:
        jury_sums_for_ranking.loc[vc, vc] = pd.NA

# 3c — Rank within each jury, 1 = highest sum.
# method='min' = tied entries share the lower rank; ties are resolved later.
jury_ranks = jury_sums_for_ranking.rank(ascending=False, method='min')

# 3d — Convert rank → Eurovision points
POINTS_SCALE = {1: 12, 2: 10, 3: 8, 4: 7, 5: 6,
                     6:  5, 7:  4, 8: 3, 9: 2, 10: 1}

def rank_to_points(rank):
    """Eurovision-style points; ranks above 10 (and NaN home rows) get 0."""
    if pd.isna(rank):
        return 0
    return POINTS_SCALE.get(int(rank), 0)

jury_points = jury_ranks.map(rank_to_points).astype(int)
jury_points

Voting_country,Croatia,Finland,Montenegro,Serbia,Sweden,Belgium,Georgia,Israel,Moldova,Poland,Estonia,Greece,Lithuania,Portugal,San Marino,Germany,Italy
Participating_country,,,,,,,,,,,,,,,,,
Croatia,0,4,10,0,3,10,12,2,12,0,5,8,4,0,0,0,8
Finland,10,0,5,8,7,8,0,7,8,8,0,1,8,10,5,12,0
Montenegro,0,12,0,2,0,12,8,0,10,2,12,7,2,3,4,1,0
Serbia,7,1,7,0,0,7,3,10,4,12,2,5,0,5,10,4,0
Sweden,1,5,6,5,0,5,0,4,5,3,0,3,0,8,2,10,3
Belgium,0,0,0,3,8,0,7,5,0,5,0,0,10,1,0,6,0
Georgia,0,2,0,7,0,6,0,1,6,7,3,6,7,0,6,5,7
Israel,8,10,0,1,2,0,0,0,2,1,4,10,0,0,0,8,2
Moldova,5,6,1,12,0,1,0,0,0,6,6,2,0,2,12,0,1


In [61]:
# 3e — Detect ties so we can resolve them in the next step.
# Each entry: (voting_country, tied_rank, [tied_participating_countries])
ties_found = []
for vc in jury_ranks.columns:
    rank_counts = jury_ranks[vc].dropna().value_counts()
    for r, n in rank_counts.items():
        if n > 1:
            tied_countries = jury_ranks.index[jury_ranks[vc] == r].tolist()
            ties_found.append((vc, int(r), tied_countries))

if ties_found:
    print(f"⚠ {len(ties_found)} tie(s) found — to be resolved by tie-breaker rules:")
    for vc, r, tied in ties_found:
        print(f"  {vc} jury, rank {r}: {tied}")
else:
    print("✓ No ties detected")

✓ No ties detected


In [62]:
# Sanity check: total points awarded by each jury
# - Participating jury with P countries awards points to P-1 outsiders
# - Non-participating jury awards points to all P countries
expected = {
    vc: sum(list(POINTS_SCALE.values())[:len(jury_ranks.index) - (vc in jury_ranks.index)])
    for vc in jury_points.columns
}
actual = jury_points.sum().to_dict()

for vc in actual:
    status = '✓' if actual[vc] == expected[vc] else '✗'
    print(f"  {status} {vc}: awarded {actual[vc]} (expected {expected[vc]})")

  ✓ Croatia: awarded 58 (expected 58)
  ✓ Finland: awarded 58 (expected 58)
  ✓ Montenegro: awarded 58 (expected 58)
  ✓ Serbia: awarded 58 (expected 58)
  ✓ Sweden: awarded 58 (expected 58)
  ✓ Belgium: awarded 58 (expected 58)
  ✓ Georgia: awarded 58 (expected 58)
  ✓ Israel: awarded 58 (expected 58)
  ✓ Moldova: awarded 58 (expected 58)
  ✓ Poland: awarded 58 (expected 58)
  ✓ Estonia: awarded 58 (expected 58)
  ✓ Greece: awarded 58 (expected 58)
  ✓ Lithuania: awarded 58 (expected 58)
  ✓ Portugal: awarded 58 (expected 58)
  ✓ San Marino: awarded 58 (expected 58)
  ✓ Germany: awarded 58 (expected 58)
  ✓ Italy: awarded 58 (expected 58)


## Step 4 — Final classification

Sum the points awarded by every jury per participating country and rank.

In [63]:
final_scores = jury_points.sum(axis=1)
final_ranks  = final_scores.rank(ascending=False, method='min').astype(int)

final_classification = (
    pd.DataFrame({'Total_points': final_scores, 'Rank': final_ranks})
    .sort_values('Rank')
)
final_classification

,Total_points,Rank
Participating_country,,
Finland,97,1
Croatia,78,2
Lithuania,78,2
Serbia,77,4
Montenegro,75,5
Portugal,72,6
San Marino,70,7
Greece,64,8
Georgia,63,9
